<a href="https://colab.research.google.com/github/Kimballa05/agentic-ai-jetstream2-bioinformatics/blob/main/agentic_ai_ecoli_wgs_workshop.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Agentic AI for *E. coli* WGS Intake, Preprocessing, and QC

## Jetstream2 + OpenCode workshop prototype

### Scenario

You have been given four public sequencing accessions that are **supposed to belong to an *Escherichia coli* whole-genome sequencing (WGS) project**. The agent should verify that assumption from NCBI metadata before processing anything, retrieve only valid WGS runs, perform pre/post sequencing QC and `fastp` preprocessing, apply a simple supervised ML quality classifier, and write a concise report.

One accession is deliberately a 16S amplicon run and should be flagged as a mismatch rather than sent through the WGS workflow.

**Each step of the workflow will provide an example prompt to run as well as explicit code that represents what should be running at that step.**

## Workflow

```text
samples.tsv
    |
    v
Agent inspects NCBI metadata
    |
    +----------------------+
    |                      |
    v                      v
valid E. coli WGS      mismatch / unclear
    |                      |
    v                      v
download reads             FLAG
    |
    v
raw FASTQ validation
    |
    v
SeqKit + FastQC + MultiQC
    |
    v
fastp preprocessing
    |
    v
post-QC
    |
    v
QC feature table
    |
    v
supervised ML: PASS / REVIEW
    |
    v
final report
```

## Workshop accessions

The expected project is paired-end *E. coli* WGS.

| sample_id | run_accession |
|---|---|
| sample01 | `ERR4567643` |
| sample02 | `SRR25073407` |
| sample03 | `SRR29154704` |
| sample04 | `SRR27836258` |

The participant-facing workflow should not identify the mismatch in advance.

In [ ]:
import pandas as pd
from pathlib import Path

samples = pd.DataFrame({
    "sample_id": ["sample01", "sample02", "sample03", "sample04"],
    "run_accession": [
        "ERR4567643",
        "SRR25073407",
        "SRR29154704",
        "SRR27836258",
    ],
})

BASE = Path("agentic_ecoli_wgs")

for d in [
    "metadata/xml",
    "raw",
    "processed",
    "qc/raw/fastqc",
    "qc/raw/multiqc",
    "qc/processed/fastqc",
    "qc/processed/multiqc",
    "reports/fastp",
    "logs",
]:
    (BASE / d).mkdir(parents=True, exist_ok=True)

for sid in samples["sample_id"]:
    (BASE / "raw" / sid).mkdir(parents=True, exist_ok=True)
    (BASE / "processed" / sid).mkdir(parents=True, exist_ok=True)

samples.to_csv(BASE / "samples.tsv", sep="\t", index=False)
samples

,sample_id,run_accession
0,sample01,ERR4567643
1,sample02,SRR25073407
2,sample03,SRR29154704
3,sample04,SRR27836258


## Project structure

```text
agentic_ecoli_wgs/
├── samples.tsv
├── metadata/
│   ├── xml/
│   ├── sra_metadata.tsv
│   ├── metadata_review.tsv
│   └── valid_wgs.tsv
├── raw/
├── processed/
├── qc/
│   ├── raw/
│   └── processed/
├── reports/
│   └── fastp/
└── logs/
```

`metadata/` contains input information, `raw/` untouched reads, `processed/` cleaned reads, `qc/` QC outputs, `reports/` summary tables and ML results, and `logs/` executed commands.

## Bioinformatics tools

While this should already be set up for everyone on Jetstream2, here is code to set up tools in this Colab.

In [ ]:
# Install a small tool set for the Colab prototype
!apt-get install -y -qq sra-toolkit fastqc fastp pigz > /dev/null
!pip install -q multiqc pandas scikit-learn pyyaml

# Step 1 — Validate the sequencing accessions

Before analyzing sequencing data, we need to verify that each accession corresponds to the expected organism and sequencing experiment. The agent will retrieve metadata from NCBI SRA and examine information such as organism, library strategy, and sequencing layout. Samples that do not meet our paired-end E. coli WGS requirements will be flagged for exclusion or manual review. This prevents unsuitable datasets from entering downstream analyses.

### Example agent prompt

Inspect the public NCBI SRA metadata for the accessions listed in samples.tsv. Review the organism, strategy, source, selection, layout, platform, instrument, experiment title, description, and available library-preparation information. Save the retrieved metadata to metadata/sra_metadata.tsv, then create metadata/metadata_review.tsv containing the sample ID, accession, whether the run is WGS-compatible, a brief evidence summary, and a status of VALID_WGS, MISMATCH, or REVIEW.

### Explicit Code: retrieve and parse NCBI SRA metadata

This uses NCBI Entrez Direct (`esearch` and `efetch`) so the workflow preserves experiment titles and descriptions in addition to structured library fields.

In [ ]:
import subprocess

RUN_METADATA_RETRIEVAL = False  # Set True on Jetstream2 when Entrez Direct is installed.

if RUN_METADATA_RETRIEVAL:
    for _, row in samples.iterrows():
        sample = row["sample_id"]
        accession = row["run_accession"]
        out_xml = BASE / "metadata" / "xml" / f"{sample}_{accession}.xml"

        cmd = (
            f'esearch -db sra -query "{accession}" '
            f'| efetch -format native -mode xml '
            f'> "{out_xml}"'
        )

        subprocess.run(cmd, shell=True, check=True)

        with open(BASE / "logs" / "commands.log", "a") as log:
            log.write(cmd + "\n")

        print(f"Saved {out_xml}")
else:
    print("Metadata retrieval scaffold ready. Set RUN_METADATA_RETRIEVAL=True on Jetstream2.")

In [ ]:
import xml.etree.ElementTree as ET

def get_text(root, path):
    el = root.find(path)
    return "" if el is None or el.text is None else el.text.strip()

def parse_sra_xml(sample, accession):
    xml_file = BASE / "metadata" / "xml" / f"{sample}_{accession}.xml"
    root = ET.parse(xml_file).getroot()

    layout_node = root.find(".//LIBRARY_LAYOUT")
    layout = ""
    if layout_node is not None and len(layout_node):
        layout = layout_node[0].tag

    run = root.find(".//RUN")
    run_accession = run.attrib.get("accession", accession) if run is not None else accession

    return {
        "sample_id": sample,
        "run_accession": run_accession,
        "organism": get_text(root, ".//SAMPLE/SAMPLE_NAME/SCIENTIFIC_NAME"),
        "strategy": get_text(root, ".//LIBRARY_DESCRIPTOR/LIBRARY_STRATEGY"),
        "source": get_text(root, ".//LIBRARY_DESCRIPTOR/LIBRARY_SOURCE"),
        "selection": get_text(root, ".//LIBRARY_DESCRIPTOR/LIBRARY_SELECTION"),
        "layout": layout,
        "instrument": get_text(root, ".//INSTRUMENT_MODEL"),
        "experiment_title": get_text(root, ".//EXPERIMENT/TITLE"),
        "experiment_description": get_text(root, ".//DESIGN_DESCRIPTION"),
    }

# Run after XML retrieval:
#
# metadata = pd.DataFrame([
#     parse_sra_xml(row.sample_id, row.run_accession)
#     for row in samples.itertuples()
# ])
#
# metadata.to_csv(BASE / "metadata" / "sra_metadata.tsv", sep="\t", index=False)
# metadata



The agent should read `metadata/sra_metadata.tsv` and create the review table itself.

In [ ]:
import re

def create_metadata_review(metadata):
    reviews = []

    for _, r in metadata.fillna("").iterrows():
        combined = " ".join([
            r["strategy"],
            r["source"],
            r["selection"],
            r["experiment_title"],
            r["experiment_description"],
        ])

        organism_ok = "escherichia coli" in r["organism"].lower()

        if (
            r["strategy"].upper() == "WGS"
            and r["source"].upper() == "GENOMIC"
            and organism_ok
        ):
            status = "VALID_WGS"
            evidence = (
                f"Organism={r['organism']}; Strategy={r['strategy']}; "
                f"Source={r['source']}; Layout={r['layout']}"
            )
        elif re.search(r"16S|V3[-–]?V4|V4|amplicon", combined, re.IGNORECASE):
            status = "MISMATCH"
            evidence = "Metadata indicates targeted 16S/amplicon sequencing rather than WGS."
        else:
            status = "REVIEW"
            evidence = "Metadata is insufficient or conflicting for automatic WGS validation."

        reviews.append({
            "sample_id": r["sample_id"],
            "run_accession": r["run_accession"],
            "wgs_compatible": status == "VALID_WGS",
            "evidence_summary": evidence,
            "status": status,
        })

    review = pd.DataFrame(reviews)
    review.to_csv(BASE / "metadata" / "metadata_review.tsv", sep="\t", index=False)
    return review

# Example:
# metadata = pd.read_csv(BASE / "metadata" / "sra_metadata.tsv", sep="\t")
# review = create_metadata_review(metadata)
# review

# Step 2 — Retrieve the validated WGS reads

Once the appropriate datasets have been identified, we can download their raw sequencing reads using the SRA Toolkit. The agent will retrieve only validated WGS samples and organize their paired-end FASTQ files into the project directory. Original sequencing files will remain unchanged to ensure reproducibility and allow reprocessing if needed.

### Example agent prompt

Using metadata/metadata_review.tsv, retrieve only samples classified as VALID_WGS with SRA Toolkit and save the untouched FASTQ files under raw/'sample_id'/. Verify that both R1 and R2 were produced and that the files are non-empty and readable. Record the retrieval commands in logs/commands.log and report the sample ID, accession, filenames, file sizes, and retrieval status. Do not download accessions classified as mismatches.

In [ ]:
def write_valid_wgs_table():
    review = pd.read_csv(BASE / "metadata" / "metadata_review.tsv", sep="\t")

    valid = review.loc[
        review["status"] == "VALID_WGS",
        ["sample_id", "run_accession"],
    ]

    valid.to_csv(BASE / "metadata" / "valid_wgs.tsv", sep="\t", index=False)
    return valid

# valid = write_valid_wgs_table()
# valid

In [ ]:
RUN_DOWNLOADS = False

if RUN_DOWNLOADS:
    valid = pd.read_csv(BASE / "metadata" / "valid_wgs.tsv", sep="\t")

    for _, row in valid.iterrows():
        sample = row["sample_id"]
        accession = row["run_accession"]
        outdir = BASE / "raw" / sample

        commands = [
            f"prefetch --max-size u {accession}",
            f"vdb-validate {accession}",
            f"fasterq-dump {accession} --split-files -e 4 -O {outdir}",
            f"pigz -p 4 {outdir}/*.fastq",
        ]

        for cmd in commands:
            print(cmd)
            subprocess.run(cmd, shell=True, check=True)
            with open(BASE / "logs" / "commands.log", "a") as log:
                log.write(cmd + "\n")
else:
    print("Download scaffold ready. Set RUN_DOWNLOADS=True after metadata validation.")

# Step 3 — Validate the raw FASTQ files and run initial QC

Raw sequencing reads may contain low-quality bases, adapter contamination, or incomplete files that can interfere with downstream analyses. We will use SeqKit, FastQC, and MultiQC to examine read counts, sequence quality, and other important metrics. The agent will also verify that paired-end files are present and readable, identifying potential problems before preprocessing begins.

### Example agent prompt

Inspect the downloaded WGS FASTQ files before preprocessing. Confirm that R1 and R2 are present, the files are non-empty and readable, the FASTQ structure appears valid, and paired files have compatible read counts. Run SeqKit, FastQC, and MultiQC, save raw QC outputs under qc/raw/, and summarize read counts, read lengths, GC content, sequence quality, Q30 percentage where available, ambiguous-base content, duplication, adapter content, and major FastQC warnings before continuing.

In [ ]:
import gzip

def validate_fastq_gz(path):
    path = Path(path)

    if not path.exists():
        return False, "missing"
    if path.stat().st_size == 0:
        return False, "empty"

    try:
        with gzip.open(path, "rt") as fh:
            header = fh.readline().strip()
            seq = fh.readline().strip()
            plus = fh.readline().strip()
            qual = fh.readline().strip()

        if not header.startswith("@") or not plus.startswith("+"):
            return False, "FASTQ structure not recognized"
        if len(seq) != len(qual):
            return False, "sequence/quality length mismatch"

    except Exception as exc:
        return False, str(exc)

    return True, "ok"

def validate_all_wgs_fastqs(valid):
    rows = []

    for _, row in valid.iterrows():
        sample = row["sample_id"]
        accession = row["run_accession"]

        r1 = BASE / "raw" / sample / f"{accession}_1.fastq.gz"
        r2 = BASE / "raw" / sample / f"{accession}_2.fastq.gz"

        r1_ok, r1_msg = validate_fastq_gz(r1)
        r2_ok, r2_msg = validate_fastq_gz(r2)

        rows.append({
            "sample_id": sample,
            "run_accession": accession,
            "r1_ok": r1_ok,
            "r1_status": r1_msg,
            "r2_ok": r2_ok,
            "r2_status": r2_msg,
        })

    result = pd.DataFrame(rows)
    result.to_csv(BASE / "reports" / "fastq_validation.tsv", sep="\t", index=False)
    return result

In [ ]:
raw_qc_commands = [
    f"seqkit stats -a -T {BASE}/raw/*/*.fastq.gz > {BASE}/qc/raw/seqkit_raw.tsv",
    f"fastqc -t 4 -o {BASE}/qc/raw/fastqc {BASE}/raw/*/*.fastq.gz",
    f"multiqc {BASE}/qc/raw/fastqc -o {BASE}/qc/raw/multiqc -f",
]

RUN_RAW_QC = False

if RUN_RAW_QC:
    for cmd in raw_qc_commands:
        print(cmd)
        subprocess.run(cmd, shell=True, check=True)
        with open(BASE / "logs" / "commands.log", "a") as log:
            log.write(cmd + "\n")
else:
    print("\n".join(raw_qc_commands))

# Step 4 — Preprocess the WGS reads

Preprocessing improves sequencing data quality by removing adapters and filtering low-quality reads. We will use fastp to clean the paired-end FASTQ files while preserving the original data. The agent will construct the preprocessing commands, organize the outputs, and document the parameters used to ensure that the workflow remains reproducible.

### Example agent prompt

Preprocess the validated paired-end WGS reads with fastp using conservative adapter detection and standard quality filtering while leaving the raw FASTQ files unchanged. Save processed reads under processed/'sample_id'/ using predictable names such as sample01_R1.clean.fastq.gz and sample01_R2.clean.fastq.gz, save the corresponding fastp HTML and JSON reports under reports/fastp/, record the exact commands used, and summarize whether preprocessing completed successfully and how many reads were retained.

In [ ]:
def build_fastp_command(sample, accession):
    r1 = BASE / "raw" / sample / f"{accession}_1.fastq.gz"
    r2 = BASE / "raw" / sample / f"{accession}_2.fastq.gz"
    out1 = BASE / "processed" / sample / f"{sample}_R1.clean.fastq.gz"
    out2 = BASE / "processed" / sample / f"{sample}_R2.clean.fastq.gz"
    html = BASE / "reports" / "fastp" / f"{sample}_fastp.html"
    js = BASE / "reports" / "fastp" / f"{sample}_fastp.json"

    return (
        f"fastp -i {r1} -I {r2} "
        f"-o {out1} -O {out2} "
        f"--detect_adapter_for_pe --thread 4 "
        f"--html {html} --json {js}"
    )

RUN_FASTP = False

if RUN_FASTP:
    valid = pd.read_csv(BASE / "metadata" / "valid_wgs.tsv", sep="\t")

    for _, row in valid.iterrows():
        cmd = build_fastp_command(row["sample_id"], row["run_accession"])
        print(cmd)

        subprocess.run(cmd, shell=True, check=True)

        with open(BASE / "logs" / "commands.log", "a") as log:
            log.write(cmd + "\n")
else:
    print("fastp scaffold ready.")

# Step 5 — Run post-processing QC and compare results

After preprocessing, we need to determine whether read quality improved without losing an excessive amount of sequencing data. We will repeat the QC analysis and compare the processed reads against the original files. The agent will help summarize changes in read retention, sequence quality, and other metrics to identify samples requiring further investigation.

### Example agent prompt

Run SeqKit, FastQC, and MultiQC on the processed reads and save the outputs under qc/processed/. Compare raw and processed data for each sample, including raw and processed read-pair counts, retention percentage, mean Phred quality, Q30 percentage, ambiguous N percentage, R1/R2 pair balance, and important FastQC changes. Summarize any sample whose preprocessing results appear substantially different from the others without automatically removing it.

In [ ]:
processed_qc_commands = [
    f"seqkit stats -a -T {BASE}/processed/*/*.fastq.gz > {BASE}/qc/processed/seqkit_processed.tsv",
    f"fastqc -t 4 -o {BASE}/qc/processed/fastqc {BASE}/processed/*/*.fastq.gz",
    f"multiqc {BASE}/qc/processed/fastqc -o {BASE}/qc/processed/multiqc -f",
]

RUN_PROCESSED_QC = False

if RUN_PROCESSED_QC:
    for cmd in processed_qc_commands:
        print(cmd)
        subprocess.run(cmd, shell=True, check=True)
        with open(BASE / "logs" / "commands.log", "a") as log:
            log.write(cmd + "\n")
else:
    print("\n".join(processed_qc_commands))

# Checkpoint — Evaluate the results: Examine the raw and processed QC reports. Did preprocessing improve read quality? Were any samples affected by substantial read loss?

# Step 6 — Build the ML QC feature table

Sequencing QC generates several metrics that can be combined to evaluate overall sample quality. In this step, we will extract measurements such as read retention, mean Phred score, and sequence length into a structured feature table. This table will serve as input for a machine-learning classifier designed to distinguish acceptable samples from those needing further review.

### Example agent prompt

Using the processed reads and post-processing QC results, create one row per WGS sample containing retention_percent, mean_phred, q30_percent, n_percent, and pair_balance_percent, along with the sample ID and accession. Save the completed table as reports/processed_qc_features.tsv, check it for missing or invalid values, and display the final feature table before running the classifier.

In [ ]:
import numpy as np

def count_reads(path):
    lines = 0
    with gzip.open(path, "rt") as fh:
        for lines, _ in enumerate(fh, start=1):
            pass
    return lines // 4

def scan_quality(path, max_reads=25000):
    reads = bases = q_sum = q30 = n_count = 0

    with gzip.open(path, "rt") as fh:
        while reads < max_reads:
            header = fh.readline()
            if not header:
                break

            seq = fh.readline().strip()
            fh.readline()
            qual = fh.readline().strip()

            if len(seq) != len(qual):
                raise ValueError(f"Sequence/quality mismatch in {path}")

            q = np.fromiter((ord(c) - 33 for c in qual), dtype=np.int16)

            reads += 1
            bases += len(seq)
            q_sum += int(q.sum())
            q30 += int((q >= 30).sum())
            n_count += seq.upper().count("N")

    if bases == 0:
        raise ValueError(f"No reads found in {path}")

    return {"bases": bases, "q_sum": q_sum, "q30": q30, "n_count": n_count}

def build_qc_feature_table(valid):
    rows = []

    for _, r in valid.iterrows():
        sample = r["sample_id"]
        accession = r["run_accession"]

        raw1 = BASE / "raw" / sample / f"{accession}_1.fastq.gz"
        raw2 = BASE / "raw" / sample / f"{accession}_2.fastq.gz"
        clean1 = BASE / "processed" / sample / f"{sample}_R1.clean.fastq.gz"
        clean2 = BASE / "processed" / sample / f"{sample}_R2.clean.fastq.gz"

        raw_r1, raw_r2 = count_reads(raw1), count_reads(raw2)
        clean_r1, clean_r2 = count_reads(clean1), count_reads(clean2)

        raw_pairs = min(raw_r1, raw_r2)
        clean_pairs = min(clean_r1, clean_r2)

        retention = 100 * clean_pairs / raw_pairs
        pair_balance = 100 * min(clean_r1, clean_r2) / max(clean_r1, clean_r2)

        q1, q2 = scan_quality(clean1), scan_quality(clean2)
        bases = q1["bases"] + q2["bases"]

        rows.append({
            "sample_id": sample,
            "run_accession": accession,
            "retention_percent": retention,
            "mean_phred": (q1["q_sum"] + q2["q_sum"]) / bases,
            "q30_percent": 100 * (q1["q30"] + q2["q30"]) / bases,
            "n_percent": 100 * (q1["n_count"] + q2["n_count"]) / bases,
            "pair_balance_percent": pair_balance,
        })

    features = pd.DataFrame(rows)
    features.to_csv(BASE / "reports" / "processed_qc_features.tsv", sep="\t", index=False)
    return features

# valid = pd.read_csv(BASE / "metadata" / "valid_wgs.tsv", sep="\t")
# features = build_qc_feature_table(valid)
# features

# Step 7 — Run the supervised QC classifier

We will demonstrate how machine learning can support automated sequencing quality assessment. Using synthetic training data, the agent will compare logistic regression and random forest models and apply the selected classifier to our processed samples. Predictions will classify samples as PASS or REVIEW, providing an additional decision-support tool rather than replacing manual QC assessment.

### Example agent prompt

Use the provided training data to compare logistic regression and random forest models with the supplied validation procedure, then apply the selected model to reports/processed_qc_features.tsv. Save the predictions to reports/ml_qc_results.tsv and report each sample's PASS or REVIEW classification, estimated probability of PASS, and associated QC measurements. Treat the model as decision support only and, for samples classified as REVIEW, identify which QC measurements appear most concerning rather than automatically excluding the sample.

### Demonstration training data

**The workshop samples are not enough to train a supervised model. For now, we'll generate a reproducible synthetic training table representing example `PASS` and `REVIEW` QC outcomes. This is a teaching classifier, not a validated production QC model.**

In [ ]:
rng = np.random.default_rng(42)

n_pass = 50
n_review = 30

pass_df = pd.DataFrame({
    "retention_percent": rng.uniform(88, 100, n_pass),
    "mean_phred": rng.uniform(32, 38, n_pass),
    "q30_percent": rng.uniform(82, 98, n_pass),
    "n_percent": rng.uniform(0.0, 0.30, n_pass),
    "pair_balance_percent": rng.uniform(99.5, 100, n_pass),
    "label": "PASS",
})

review_df = pd.DataFrame({
    "retention_percent": rng.uniform(35, 90, n_review),
    "mean_phred": rng.uniform(22, 32, n_review),
    "q30_percent": rng.uniform(40, 84, n_review),
    "n_percent": rng.uniform(0.2, 4.0, n_review),
    "pair_balance_percent": rng.uniform(92, 100, n_review),
    "label": "REVIEW",
})

training = pd.concat([pass_df, review_df], ignore_index=True)
training.sample(8, random_state=1)

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate

FEATURES = [
    "retention_percent",
    "mean_phred",
    "q30_percent",
    "n_percent",
    "pair_balance_percent",
]

X = training[FEATURES]
y = training["label"]

models = {
    "logistic_regression": Pipeline([
        ("scale", StandardScaler()),
        ("model", LogisticRegression(random_state=42)),
    ]),
    "random_forest": RandomForestClassifier(
        n_estimators=200,
        class_weight="balanced",
        random_state=42,
    ),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

rows = []
for name, model in models.items():
    scores = cross_validate(
        model,
        X,
        y,
        cv=cv,
        scoring=["accuracy", "precision_macro", "recall_macro", "f1_macro"],
    )

    rows.append({
        "model": name,
        "accuracy": scores["test_accuracy"].mean(),
        "precision_macro": scores["test_precision_macro"].mean(),
        "recall_macro": scores["test_recall_macro"].mean(),
        "f1_macro": scores["test_f1_macro"].mean(),
    })

comparison = pd.DataFrame(rows).sort_values("f1_macro", ascending=False)
comparison.to_csv(BASE / "reports" / "model_comparison.tsv", sep="\t", index=False)
comparison

In [ ]:
best_name = comparison.iloc[0]["model"]
best_model = models[best_name]
best_model.fit(X, y)

qc = pd.read_csv(BASE / "reports" / "processed_qc_features.tsv", sep="\t")

qc["ml_qc_class"] = best_model.predict(qc[FEATURES])

classes = list(best_model.classes_)
pass_index = classes.index("PASS")
qc["pass_probability"] = best_model.predict_proba(qc[FEATURES])[:, pass_index]

qc.to_csv(BASE / "reports" / "ml_qc_results.tsv", sep="\t", index=False)
qc

# Step 8 — Generate the final report

The final step consolidates the workflow results into a reproducible report. The agent will summarize metadata validation, preprocessing outcomes, QC metrics, and machine-learning classifications for each submitted sample. This report provides a record of what was performed, which samples passed evaluation, and which may require further attention.

### Example agent prompt

Create reports/workflow_report.md summarizing every submitted accession, including its metadata-validation status, evidence for WGS compatibility or mismatch, retrieval status, preprocessing performed, raw and processed read counts, retention percentage, major QC findings, ML classification, PASS probability, final status, and relevant output locations. Also save software versions to reports/software_versions.txt and maintain the command history in logs/commands.log. End the report by identifying which WGS samples appear ready for downstream analysis, which require review, and which accessions were rejected before processing because they did not match the intended WGS experiment.

In [ ]:
metadata_review = pd.read_csv(BASE / "metadata" / "metadata_review.tsv", sep="\t")
ml_results = pd.read_csv(BASE / "reports" / "ml_qc_results.tsv", sep="\t")

final_summary = metadata_review.merge(
    ml_results,
    on=["sample_id", "run_accession"],
    how="left",
)

final_summary.to_csv(
    BASE / "reports" / "final_summary.tsv",
    sep="\t",
    index=False,
)

final_summary

## Expected final outputs

```text
metadata/sra_metadata.tsv
metadata/metadata_review.tsv
metadata/valid_wgs.tsv

raw/<sample_id>/*.fastq.gz
processed/<sample_id>/*.clean.fastq.gz

qc/raw/
qc/processed/

reports/fastp/
reports/fastq_validation.tsv
reports/processed_qc_features.tsv
reports/model_comparison.tsv
reports/ml_qc_results.tsv
reports/final_summary.tsv
reports/software_versions.txt
reports/workflow_report.md

logs/commands.log
```


## Future downstream analyses

This workshop (for now) stops after WGS validation, preprocessing, post-processing QC, and ML-based quality assessment, but the processed reads could serve as the starting point for a more complete bacterial genomics workflow. Possible downstream analyses:

1.   de novo assembly (SPAdes)
2.   genome annotation (Bakta)
3.   antimicrobial resistance detection with AMRFinderPlus
4.   Other options people may think of



# Instructor answer key

Remove this section from the participant-facing version.

`ERR4567643` → valid *E. coli* WGS  
`SRR25073407` → valid *E. coli* WGS  
`SRR29154704` → valid *E. coli* WGS  
`SRR27836258` → deliberate 16S / non-WGS mismatch; flag before download and preprocessing